## Local variance on earth embeddings 

In the present notebook we conduct an analysis to understand hoe embeddings vary from each other in a local delimited region. 

In [ ]:
# Import necessary libraries 
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from itertools import combinations
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

import rasterio
import polars as pl
from rasterio.plot import show
from pyproj import CRS


# Internal Libraries 
from pathlib import Path
import sys

PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))


from src.utils.similarity import group_cosine_stats
from src.utils.geom import split_streets_at_intersections

# Parameters 

VECTOR_FILE_PATH = "../data/clean/mun_003_2024.tif"
FIGURES_PATH = "../docs/resources/embedding_showcase/earth_embedings_sim_local_var_"
PD_SHP_PATH = "../data/spatial/pd/09mun.shp"
PD_MZA_SHP_PATH = "../data/spatial/pd/09m.shp"
PD_AGEB_SHP_PATH = "../data/spatial/pd/09a.shp"
PD_STREET_SHP_PATH = "../data/spatial/pd/09e.shp"

# Language settings.
# LANG  -> default language, the one rendered inline while the notebook runs.
# LANGS -> every text-bearing figure is saved once per language on each run,
#          so a single pass produces the complete bilingual set.
LANG = "esp"
LANGS = ("esp", "eng")

Functions and utilities 

In [ ]:
# --- Bilingual figure helpers -------------------------------------------------
# Only titles/labels differ between languages: the data, colours and layout of
# every figure are identical. Spanish files carry the `_esp` suffix so they
# never overwrite their English counterpart.


def fig_path(name: str, lang: str) -> str:
    """Resource path for figure ``name`` in ``lang``."""
    suffix = "_esp" if lang == "esp" else ""
    return f"{FIGURES_PATH}{name}{suffix}.png"


def save_fig(fig, name: str, lang: str) -> str:
    """Save ``fig`` under the language-specific name and return the path."""
    path = fig_path(name, lang)
    fig.savefig(path, dpi=300, bbox_inches="tight", pad_inches=0)
    return path


def finish(fig, name: str, lang: str) -> None:
    """Save the figure, then show it only for the default language."""
    save_fig(fig, name, lang)
    if lang == LANG:
        plt.show()
    else:
        plt.close(fig)


TEXTS = {
    "esp": {
        # Area of interest
        "aoi_title": "Coyoacán, Ciudad de México\n$\\it{AdI\\ Delimitada\\ en\\ azul.}$",
        # Distribution panels (means / medians per aggregation unit)
        "dist_hist_x": "Similitud coseno media por CVEGEO",
        "dist_hist_y": "Frecuencia",
        "dist_hist_title": "Distribución de la media de similitudes",
        "dist_cdf_x": "Similitud coseno",
        "dist_cdf_y": "CDF",
        "dist_cdf_title": "Distribución acumulada",
        "dist_box_y": "Similitud coseno",
        "dist_box_title": "Resumen de estadísticas de la distribución",
        "dist_mean": "Media",
        "dist_median": "Mediana",
        "dist_stats_header": "Estadísticas nivel CVEGEO:",
        "dist_mean_of_means": "Media de medias",
        "dist_median_of_means": "Mediana de medias",
        # Choropleths
        "heat_legend": "Similitud coseno media",
        "heat_title_cvegeo": "Media de la similitud coseno uno a uno en ráster de 10 m \nDentro de CVEGEO",
        "heat_title_ageb": "Media de la similitud coseno uno a uno en ráster de 10 m \nDentro de AGEB",
        # Street coverage
        "coverage_title": "Selección de píxeles para la cobertura de calles",
        "coverage_pixels": "Píxeles",
        "coverage_street": "Calle",
        # Similarity-to-reference animation
        "gif_title": "Similitud respecto al punto de referencia (árbol verde)  \nSimilitud ≥ {thr:.2f}",
        # Year-over-year similarity distribution
        "single_hist_x": "Similitud coseno",
        "single_hist_y": "Frecuencia",
        "single_hist_title": "Distribución de similitudes coseno",
        "single_cdf_title": "Distribución acumulada",
        "single_box_title": "Resumen de estadísticas",
        "single_box_label": "Similitud",
        "single_stats_header": "Estadísticas de similitud coseno:",
        "single_mean": "Similitud media",
        "single_median": "Similitud mediana",
        # Low-similarity clusters
        "lowsim_title": (
            "Coyoacán, Ciudad de México\n"
            "$\\it{Areas\\ de\\ baja\\ similaridad\\ coseno\\ (puntos \\ resaltados, sim < 0.85).}$"
        ),
    },
    "eng": {
        "aoi_title": "Coyoacán, Mexico City\n$\\it{Delimited\\ AoI\\ in\\ blue.}$",
        "dist_hist_x": "Mean Cosine Similarity per CVEGEO",
        "dist_hist_y": "Frequency",
        "dist_hist_title": "Distribution of Mean Similarities",
        "dist_cdf_x": "Cosine Similarity",
        "dist_cdf_y": "CDF",
        "dist_cdf_title": "Cumulative Distribution",
        "dist_box_y": "Cosine Similarity",
        "dist_box_title": "Summary Statistics Distribution",
        "dist_mean": "Mean",
        "dist_median": "Median",
        "dist_stats_header": "CVEGEO-level Statistics:",
        "dist_mean_of_means": "Mean of means",
        "dist_median_of_means": "Median of means",
        "heat_legend": "Mean Cosine Similarity",
        "heat_title_cvegeo": "Mean pairwise 10m raster Cosine Similarity \nWithin CVEGEO",
        "heat_title_ageb": "Mean pairwise 10m raster Cosine Similarity \nWithin AGEB",
        "coverage_title": "Pixel selection for Street coverage",
        "coverage_pixels": "Pixels",
        "coverage_street": "Street",
        "gif_title": "Similarity to reference point (green tree)  \nSimilarity ≥ {thr:.2f}",
        "single_hist_x": "Cosine Similarity",
        "single_hist_y": "Frequency",
        "single_hist_title": "Distribution of Cosine Similarities",
        "single_cdf_title": "Cumulative Distribution",
        "single_box_title": "Summary Statistics",
        "single_box_label": "Similarity",
        "single_stats_header": "Cosine Similarity Statistics:",
        "single_mean": "Mean similarity",
        "single_median": "Median similarity",
        "lowsim_title": (
            "Coyoacán, Mexico City\n"
            "$\\it{Low\\ cosine\\ similarity\\ areas\\ (buffered\\ points, sim < 0.85).}$"
        ),
    },
}


def plot_sim_distribution(stats_df: pd.DataFrame, means: np.array,
                          medians: np.array, lang: str = "esp"):
    """Distributional analysis of means and medians for one aggregation level.

    Single code path: only the strings pulled from ``TEXTS`` change with
    ``lang``.
    """
    t = TEXTS[lang]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Mean similarity distribution
    axes[0].hist(means, bins=30, alpha=0.7, edgecolor='black', fill='Blues')
    axes[0].set_xlabel(t["dist_hist_x"])
    axes[0].set_ylabel(t["dist_hist_y"])
    axes[0].set_title(t["dist_hist_title"])
    axes[0].grid(alpha=0.3)

    # CDF
    sorted_means = np.sort(means)
    axes[1].plot(sorted_means, np.arange(1, len(sorted_means) + 1) / len(sorted_means),
                 lw=2, label=t["dist_mean"], c="steelblue")
    sorted_medians = np.sort(medians)
    axes[1].plot(sorted_medians, np.arange(1, len(sorted_medians) + 1) / len(sorted_medians),
                 lw=2, label=t["dist_median"], c="green")
    axes[1].set_xlabel(t["dist_cdf_x"])
    axes[1].set_ylabel(t["dist_cdf_y"])
    axes[1].set_title(t["dist_cdf_title"])
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    # Box plot
    axes[2].boxplot([means, medians], labels=[t["dist_mean"], t["dist_median"]])
    axes[2].set_ylabel(t["dist_box_y"])
    axes[2].set_title(t["dist_box_title"])
    axes[2].grid(alpha=0.3)

    plt.tight_layout()

    if lang == LANG:                       # print the summary once, not per language
        print(t["dist_stats_header"])
        print(stats_df.describe())
        print(f"\n{t['dist_mean_of_means']}: {means.mean():.4f}")
        print(f"{t['dist_median_of_means']}: {np.median(means):.4f}")

    return fig

### Data Extraction 

In [ ]:
# Get the AoI shape file for clipping embeddings 
# in this case CVE_MUN == '003' for filter out the AoI region 
pd_gpd = gpd.read_file(PD_SHP_PATH).query("CVE_MUN == '003' ").to_crs("WGS84")

In [ ]:
# Visualize the Area of interest 

import contextily as ctx

for _lang in LANGS:
    t = TEXTS[_lang]

    ax = pd_gpd.plot(
         color="#04506e", alpha=0.2,
        edgecolor="#03304f", linewidth=2, figsize=(10, 7)
    )

    ax.set_xlim(-99.22, -99.08)
    ax.set_ylim(19.27, 19.38)

    ctx.add_basemap(ax, crs="EPSG:4326", source=ctx.providers.CartoDB.Positron)
    ax.set_axis_off()

    ax.set_title(t["aoi_title"], fontsize=14, pad=0, loc="left")

    finish(ax.get_figure(), "aoi", _lang)

Extract the embeddings for that particular region and visualize some vector features:

In [ ]:
# Read GeoTIFF with full georeferencing
with rasterio.open(VECTOR_FILE_PATH) as src:
    data = src.read()  # Shape: (bands, height, width)
    meta = src.meta
    crs = src.crs
    bounds = src.bounds
    transform = src.transform
    
print(f"CRS: {crs}, Bounds: {bounds}")
print(f"Data shape: {data.shape}")

# Visualize first 3 components 
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i,cmap in enumerate(['viridis', 'plasma','Blues' ]):
    axes[i].imshow(data[i], cmap=cmap)
    axes[i].set_title(f'A{str(i).zfill(2)}')
plt.show()

fig.savefig(FIGURES_PATH + "emb_sample.png", dpi=300, bbox_inches="tight", pad_inches=0)

#### Dimensionality Reduction via PCA 

In [ ]:
import rasterio
import polars as pl
import numpy as np

# Load data
with rasterio.open(VECTOR_FILE_PATH) as src:
    data = src.read()  # (64, H, W)

H, W = data.shape[1:]

# Create spatial grid indices
rows, cols = np.meshgrid(np.arange(H), np.arange(W), indexing='ij')

df = pd.DataFrame({
    'row': rows.ravel(),
    'col': cols.ravel(),
    **{f'A{str(i).zfill(2)}': data[i].ravel() for i in range(data.shape[0])}
}).set_index(['row', 'col'])

# Access pixel at (10, 20)
df.loc[(10, 20)]  # Returns 64-dim series

df = df.reset_index().assign(label = lambda x:x['row'].astype(str) + '_' + x['col'].astype(str))

# Copy from original
df_ = df.copy()

# Drop na 
df.dropna(inplace=True)

print(df.head())
print(f"Shape: {df.shape}")  # (H*W, 66) - 2 coords + 64 bands


In [ ]:
# Components 
K = 7

# Extract embedding matrix — A00–A63
band_cols         = [f'A{i:02d}' for i in range(64)]
labels            = df['label'].values
embeddings        = df[band_cols].values                  # shape (n, 64)

# Normalize (despite already been normalize by construction)
scaler            = StandardScaler()
embeddings_scaled = scaler.fit_transform(embeddings)

# PCA — K principal components
pca               = PCA(n_components=K)
components        = pca.fit_transform(embeddings_scaled)  # shape (n, K)
var_explained     = pca.explained_variance_ratio_

for i in range(K):
    print(f'PC{i+1} explained variance: {var_explained[i]:.2%}')

print(f'Total explained @{K}:        {sum(var_explained):.2%}')

In [ ]:
# We added to each entry its projection 
for i in range(K):
    df[f"PC{i+1}"] =  components[:, i]

In [ ]:
# Create a components final DataFrame 
df_components = (
    df_
    .filter(["row", "col", "label"])
    .merge(
        df.filter(["label","PC1", "PC2", "PC3", "PC4", "PC5", "PC6", "PC7"])
        , on = "label", how = "left"
    )

)
# Reconstruct the raster 
raster_reconstructed = np.array([
    df_components[f'PC{i+1}'].to_numpy().reshape(H, W) 
    for i in range(K)
])


In [ ]:
m = int(np.ceil(K / 3))  # Number of columns for 3 rows

fig, axes = plt.subplots(3, m, figsize=(4*m, 12))
axes = axes.flatten()

for i in range(K):
    im = axes[i].imshow(raster_reconstructed[i], cmap='Blues')
    axes[i].set_title(f'PC{i+1}', fontsize=10)
    axes[i].axis('off')
    plt.colorbar(im, ax=axes[i], fraction=0.046, pad=0.04)

# Hide unused subplots
for i in range(K, len(axes)):
    axes[i].axis('off')

plt.tight_layout()
plt.show()

fig.savefig(FIGURES_PATH + "pca_comp.png", dpi=300, bbox_inches="tight", pad_inches=0)

In [ ]:
len(df)

## Similarity between embeddings for different aggregation Levels 

### Understanding neighborhood level differences 


Given that analyze embeddings at a $10m$ granularity and a district level is almost intractable in terms of benchmarking and comparing differences given the amount of pairwise combinations to compare ($C^{N}_{2}$ with $N > 500k$), lets work on a neighbohood level comparison. 

In [ ]:
# Read the political division section at AGEB level 
pd_mza_gdf = gpd.read_file(PD_MZA_SHP_PATH).query("CVE_MUN == '003'")

In [ ]:

# Library for GeoTIFF
import rioxarray
from shapely.geometry import Point

# Rename CVEGEO values 
gdf = pd_mza_gdf

# Load and reproject
rds = rioxarray.open_rasterio(VECTOR_FILE_PATH)
rds_projected = rds.rio.reproject(gdf.crs)

# Extract pixel centers
xx, yy = np.meshgrid(rds_projected.x.values, rds_projected.y.values)
points = gpd.GeoSeries([Point(x, y) for x, y in zip(xx.flat, yy.flat)], crs=gdf.crs)

# Spatial join
pixel_gdf = gpd.GeoDataFrame(geometry=points)
result = gpd.sjoin(pixel_gdf, gdf[["CVEGEO", "geometry"]], how="left", predicate="within")

# Reshape: (bands, height, width) → (height*width, bands)
n_bands, h, w = rds_projected.shape
pixel_data = rds_projected.values.reshape(n_bands, -1).T
pixel_df = pd.DataFrame(pixel_data, columns=[f"A{str(i+1).zfill(2)}" for i in range(n_bands)])

# Merge
output = pd.concat([pixel_df, result[["CVEGEO"]].reset_index(drop=True)], axis=1).dropna()

In [ ]:

output = output.dropna().set_index('CVEGEO')
feature_cols = [col for col in output.columns if col.startswith('A')]

cvegeo_stats = {}
for cvegeo, group in output.groupby(level='CVEGEO'):
    stats = group_cosine_stats(group[feature_cols].values)
    if stats is not None:
        cvegeo_stats[cvegeo] = stats

stats_df = pd.DataFrame(cvegeo_stats).T

means = stats_df['mean'].values
medians = stats_df['median'].values

# Plot distributions of aggregated metrics — saved in both languages
for _lang in LANGS:
    fig = plot_sim_distribution(stats_df, means, medians, lang=_lang)
    finish(fig, "mza_level_cos_sim_mean", _lang)

In [ ]:
# Merge values at ageb level with respective geometry 
stats_gdf = gpd.GeoDataFrame(
    pd_mza_gdf.merge(
    stats_df.reset_index().rename(columns={"index":"CVEGEO"}), how = "left", on = "CVEGEO"
    )
    , crs = pd_mza_gdf.crs, geometry="geometry"
    )

In [ ]:
# Import map context 
import contextily as ctx

# Reproject to Web Mercator for basemap
stats_gdf_4326 = stats_gdf.to_crs(4326)
stats_gdf_meters = stats_gdf.to_crs(pd_mza_gdf.crs)

for _lang in LANGS:
    t = TEXTS[_lang]

    fig, ax = plt.subplots(figsize=(14, 10))

    # Plot with colormap and legend
    stats_gdf_meters.plot(
        column="mean",
        cmap="Blues",
        ax=ax,
        legend=True,
        legend_kwds={"label": t["heat_legend"], "shrink": 0.8},
        alpha=0.8,
        edgecolor='black',
        linewidth=0.5
    )

    # Add CartoDB Positron basemap
    ctx.add_basemap(
        ax,
        source=ctx.providers.CartoDB.Positron,
        zoom=10,
        alpha=0.5
    )

    ax.set_title(t["heat_title_cvegeo"], fontsize=14, fontweight='bold')
    ax.set_axis_off()

    plt.tight_layout()

    finish(fig, "mza_level_cos_sim_heatmap", _lang)

### Analyze Cosine Similarity Within Street Level 

Now, let's extend the analysis to street objects, in order to understand how multiple Objects of interest maintain the similarity in terms of embedding values within them. 

In [ ]:
# Read the political division section at AGEB level 

streets_gdf = (                                                                                                                    
    gpd
    .read_file(PD_STREET_SHP_PATH)
    .query("CVE_MUN == '003'")
    .assign(STREET_ID = lambda x: x["CVEGEO"] + x["CVE_ENT"] + x["CVE_MUN"] + x["CVE_LOC"] + x["CVEVIAL"] + x["CVESEG"])           
    .drop(columns=["CVEGEO", "CVE_ENT", "CVE_MUN", "CVE_LOC", "CVEVIAL", "CVESEG"])                                                
    )                                                                                                                              

                                                                                                                                                                                                                                                              
# Split streets into corners 
streets_split = split_streets_at_intersections(                                                                                    
    streets_gdf,                                                                                                                   
    id_col="STREET_ID"                                                                                                             
).rename(columns={"STREET_ID":"CVEGEO"})



In [ ]:
# Library for GeoTIFF

from rasterio.features import shapes
from shapely.geometry import box
import rioxarray
from shapely.geometry import Point


# Rename CVEGEO values 
gdf = streets_split

# Load and reproject
rds = rioxarray.open_rasterio(VECTOR_FILE_PATH)
rds_projected = rds.rio.reproject(gdf.crs)

# Extract pixel centers
xx, yy = np.meshgrid(rds_projected.x.values, rds_projected.y.values)
points = gpd.GeoSeries([Point(x, y) for x, y in zip(xx.flat, yy.flat)], crs=gdf.crs)


x = rds_projected.x.values
y = rds_projected.y.values

dx = abs(x[1] - x[0])
dy = abs(y[1] - y[0])

polygons = [
    box(xc - dx/2, yc - dy/2,
        xc + dx/2, yc + dy/2)
    for yc in y
    for xc in x
]

pixel_gdf = gpd.GeoDataFrame(
    geometry=polygons,
    crs=gdf.crs
)

result = gpd.sjoin(
    pixel_gdf,
    gdf[["CVEGEO", "geometry"]],
    how="inner",
    predicate="intersects"
)

# Reshape: (bands, height, width) → (height*width, bands)
n_bands, h, w = rds_projected.shape
pixel_data = rds_projected.values.reshape(n_bands, -1).T
pixel_df = pd.DataFrame(pixel_data, columns=[f"A{str(i+1).zfill(2)}" for i in range(n_bands)])

# Merge
output = pd.concat([pixel_df, result[["CVEGEO"]].reset_index(drop=True)], axis=1).dropna()

In [ ]:
# Predefine consine_simmilarity functions 
from sklearn.metrics.pairwise import cosine_similarity

output = output.set_index('CVEGEO')
feature_cols = [col for col in output.columns if col.startswith('A')]

# Aggregate cosine similarity to CVEGEO level
cvegeo_stats = {}

for cvegeo, group in output.groupby(level='CVEGEO'):
    if len(group) < 2:
        continue
    
    X_group = group[feature_cols].values
    sim_matrix = cosine_similarity(X_group)
    upper_tri = sim_matrix[np.triu_indices_from(sim_matrix, k=1)]
    
    cvegeo_stats[cvegeo] = {
        'mean': upper_tri.mean(),
        'median': np.median(upper_tri),
        'std': upper_tri.std(),
        'n_pairs': len(upper_tri)
    }

stats_df = pd.DataFrame(cvegeo_stats).T
means = stats_df['mean'].values
medians = stats_df['median'].values

# Plot distributions of aggregated metrics — saved in both languages
for _lang in LANGS:
    fig = plot_sim_distribution(stats_df, means, medians, lang=_lang)
    finish(fig, "street_level_cos_sim_descriptive", _lang)

In [ ]:
# Merge values at ageb level with respective geometry 
stats_gdf = gpd.GeoDataFrame(
    streets_split.merge(
    stats_df.reset_index().rename(columns={"index":"CVEGEO"}), how = "left", on = "CVEGEO"
    )
    , crs = streets_split.crs, geometry="geometry"
    )

In [ ]:
# Import map context 
import contextily as ctx

# Reproject to Web Mercator for basemap
stats_gdf_4326 = stats_gdf.to_crs(4326)
stats_gdf_meters = stats_gdf.to_crs(streets_split.crs)

for _lang in LANGS:
    t = TEXTS[_lang]

    fig, ax = plt.subplots(figsize=(14, 10))

    # Plot with colormap and legend
    stats_gdf_meters.plot(
        column="mean",
        cmap="Blues",
        ax=ax,
        legend=True,
        legend_kwds={"label": t["heat_legend"], "shrink": 0.8},
        alpha=0.8,
        linewidth=1
    )

    # Add CartoDB Positron basemap
    ctx.add_basemap(
        ax,
        source=ctx.providers.CartoDB.Positron,
        zoom=10,
        alpha=0.5
    )

    ax.set_title(t["heat_title_cvegeo"], fontsize=14, fontweight='bold')
    ax.set_axis_off()

    plt.tight_layout()

    finish(fig, "street_level_cos_sim_heatmap", _lang)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

cvegeo = "0900300010900300010151500001_0001"

for _lang in LANGS:
    t = TEXTS[_lang]

    fig, ax = plt.subplots(figsize=(8, 8))

    # Pixels
    result.query("CVEGEO == @cvegeo").to_crs(4326).plot(
        ax=ax,
        color="white",
        edgecolor="black",
        alpha=0.35,
    )

    # Street
    streets_split.query("CVEGEO == @cvegeo").buffer(0.5).to_crs(4326).plot(
        ax=ax,
        color="gray",
        alpha=0.3,
    )

    # Custom legend
    legend_handles = [
        Patch(facecolor="white", edgecolor="black", alpha=0.35, label=t["coverage_pixels"]),
        Patch(facecolor="gray", edgecolor="black", alpha=0.3, label=t["coverage_street"]),
    ]

    ax.legend(handles=legend_handles, loc="upper right", frameon=True)

    ax.set_aspect("equal")
    ax.set_axis_off()
    ax.set_title(t["coverage_title"])

    finish(fig, "street_level_coverage", _lang)

### Analyze Cosine Similarity Within AGEB Level 

It is important to review the variance between units on AGEB levels. 

In [ ]:
# Read the political division section at AGEB level 
pd_ageb_gdf = gpd.read_file(PD_AGEB_SHP_PATH).query("CVE_MUN == '003'").to_crs(4326)

# Library for GeoTIFF
import rioxarray
from shapely.geometry import Point

# Rename CVEGEO values 
gdf = pd_ageb_gdf

# Load and reproject
rds = rioxarray.open_rasterio(VECTOR_FILE_PATH)
rds_projected = rds.rio.reproject(gdf.crs)

# Extract pixel centers
xx, yy = np.meshgrid(rds_projected.x.values, rds_projected.y.values)
points = gpd.GeoSeries([Point(x, y) for x, y in zip(xx.flat, yy.flat)], crs=gdf.crs)

# Spatial join
pixel_gdf = gpd.GeoDataFrame(geometry=points)
result = gpd.sjoin(pixel_gdf, gdf[["CVEGEO", "geometry"]], how="left", predicate="within")

# Reshape: (bands, height, width) → (height*width, bands)
n_bands, h, w = rds_projected.shape
pixel_data = rds_projected.values.reshape(n_bands, -1).T
pixel_df = pd.DataFrame(pixel_data, columns=[f"A{str(i+1).zfill(2)}" for i in range(n_bands)])

# Merge
output = pd.concat([pixel_df, result[["CVEGEO"]].reset_index(drop=True)], axis=1).dropna()


In [ ]:

output = output.dropna().set_index('CVEGEO')
feature_cols = [col for col in output.columns if col.startswith('A')]

cvegeo_stats = {}
for cvegeo, group in output.groupby(level='CVEGEO'):
    stats = group_cosine_stats(group[feature_cols].values)
    if stats is not None:
        cvegeo_stats[cvegeo] = stats

stats_df = pd.DataFrame(cvegeo_stats).T

means = stats_df['mean'].values
medians = stats_df['median'].values

# Plot distributions of aggregated metrics — saved in both languages
for _lang in LANGS:
    fig = plot_sim_distribution(stats_df, means, medians, lang=_lang)
    finish(fig, "ageb_level_cos_sim_mean", _lang)

In [ ]:
# Merge values at ageb level with respective geometry 
stats_gdf = gpd.GeoDataFrame(
    pd_ageb_gdf.merge(
    stats_df.reset_index().rename(columns={"index":"CVEGEO"}), how = "left", on = "CVEGEO"
    )
    , crs = pd_ageb_gdf.crs, geometry="geometry"
    )

In [ ]:
# Import map context 
import contextily as ctx

# Reproject to Web Mercator for basemap
stats_gdf_4326 = stats_gdf.to_crs(4326)
stats_gdf_meters = stats_gdf.to_crs(pd_ageb_gdf.crs)

for _lang in LANGS:
    t = TEXTS[_lang]

    fig, ax = plt.subplots(figsize=(14, 10))

    # Plot with colormap and legend
    stats_gdf_meters.plot(
        column="mean",
        cmap="Blues",
        ax=ax,
        legend=True,
        legend_kwds={"label": t["heat_legend"], "shrink": 0.8},
        alpha=0.8,
        edgecolor='black',
        linewidth=0.5
    )

    # Add CartoDB Positron basemap
    ctx.add_basemap(
        ax,
        source=ctx.providers.CartoDB.Positron,
        zoom=10,
        alpha=0.5
    )

    ax.set_title(t["heat_title_ageb"], fontsize=14, fontweight='bold')
    ax.set_axis_off()

    plt.tight_layout()

    finish(fig, "ageb_level_cos_sim_heatmap", _lang)

### Cosine Similarity for Layer detections 

We use cosine similarity and a point $i$ as reference ($x_i$ embedding) to classify similar entities (look-a-likes) based on threshold selection. 

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))

from src.etl.assign_point_to_geometry import read_point_data

gdf = read_point_data("../data/proc/embeddings/alpha_earth/cdmx/year=2022/", file_format="parquet").query("CVE_MUN == 3")

In [ ]:
REFERENCE_POINT = Point(-99.17095062289278, 19.353643101281044) # Middle point in Viveros Coyoacan 
THRESHOLD_RANGE = [0.75,1]
feature_cols = [f"A{str(i).zfill(2)}" for i in range(64)]

In [ ]:
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
from sklearn.metrics.pairwise import cosine_similarity


def sim_ranking_filter(
    gdf: gpd.GeoDataFrame,
    reference_point: Point,
    feature_cols: list,
):

    # Find nearest geometry
    nearest_idx = gdf.geometry.distance(reference_point).idxmin()

    # Convert to numeric arrays
    X = gdf[feature_cols].astype(float).to_numpy()
    x_ref = (
        gdf.loc[[nearest_idx], feature_cols]
        .astype(float)
        .to_numpy()
    )

    similarities = cosine_similarity(X, x_ref).flatten()

    return (
        gdf.assign(similarity_to_ref=similarities)
           .copy()
    )

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os

vmin = 0
vmax = 1

thresholds = np.linspace(0.95, 0.80, 15)

gdf_sims = sim_ranking_filter(
        gdf,
        REFERENCE_POINT,
        feature_cols,
    ).to_crs(3857)

import io

import contextily as ctx
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
from PIL import Image as PILImage

# Helpers

def _fig_to_pil(fig: Figure, dpi: int = 130) -> PILImage.Image:
    buf = io.BytesIO()
    fig.savefig(
        buf,
        format="png",
        dpi=dpi,
        bbox_inches="tight",
        facecolor="white",
    )
    buf.seek(0)
    img = PILImage.open(buf).copy()
    plt.close(fig)
    return img


def _frames_to_gif(frames: list[PILImage.Image], duration: int = 250) -> bytes:
    buf = io.BytesIO()
    frames[0].save(
        buf,
        format="GIF",
        save_all=True,
        append_images=frames[1:],
        duration=duration,
        loop=0,
        optimize=True,
    )
    return buf.getvalue()


# Build one GIF per language — only the title text differs

GIF_PATH = "../docs/figures/similarity_greens_test"

for _lang in LANGS:
    t = TEXTS[_lang]
    frames = []

    for thr in thresholds:

        gdf_filtered = gdf_sims.query("similarity_to_ref >= @thr")

        fig, ax = plt.subplots(figsize=(8, 8))

        # Similar points
        gdf_filtered.plot(
            ax=ax,
            column="similarity_to_ref",
            cmap="Greens",
            vmin=vmin,
            vmax=vmax,
            markersize=0.75,     
            alpha=0.25,
            legend=False,
            zorder=3,
        )

        # Carto Positron basemap
        ctx.add_basemap(
            ax,
            source=ctx.providers.CartoDB.Positron,
            attribution=True,
        )

        # Reference point
        ax.scatter(
            REFERENCE_POINT.x,
            REFERENCE_POINT.y,
            color="black",
            marker="+",
            s=13,
            zorder=10,
        )

        ax.set_title(t["gif_title"].format(thr=thr))
        ax.set_axis_off()

        frames.append(_fig_to_pil(fig))

    # Save GIF
    gif_bytes = _frames_to_gif(frames, duration=250)

    suffix = "_esp" if _lang == "esp" else ""
    with open(f"{GIF_PATH}{suffix}.gif", "wb") as f:
        f.write(gif_bytes)


### Systemic Shocks in Earth Embeddings 

In this section we conduct a change analysis over the year over year vector embeddings under the same area. 

In [ ]:
def plot_sim_distribution_single(similarities: np.array, lang: str = "esp"):
    """Distributional analysis for a single array of cosine similarities.

    Renamed from ``plot_sim_distribution`` so it no longer shadows the
    aggregation-level function defined at the top of the notebook, which takes
    a different signature.
    """
    t = TEXTS[lang]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Histogram
    axes[0].hist(
        similarities,
        bins=30,
        alpha=0.7,
        edgecolor="black",
        color="steelblue",
    )
    axes[0].set_xlabel(t["single_hist_x"])
    axes[0].set_ylabel(t["single_hist_y"])
    axes[0].set_title(t["single_hist_title"])
    axes[0].grid(alpha=0.3)

    # CDF
    sorted_sim = np.sort(similarities)
    axes[1].plot(
        sorted_sim,
        np.arange(1, len(sorted_sim) + 1) / len(sorted_sim),
        lw=2,
        color="steelblue",
    )
    axes[1].set_xlabel(t["single_hist_x"])
    axes[1].set_ylabel(t["dist_cdf_y"])
    axes[1].set_title(t["single_cdf_title"])
    axes[1].grid(alpha=0.3)

    # Box plot
    axes[2].boxplot(
        similarities,
        labels=[t["single_box_label"]],
    )
    axes[2].set_ylabel(t["single_hist_x"])
    axes[2].set_title(t["single_box_title"])
    axes[2].grid(alpha=0.3)

    plt.tight_layout()

    if lang == LANG:                       # print the summary once, not per language
        print(t["single_stats_header"])
        print(pd.Series(similarities).describe())
        print(f"\n{t['single_mean']}: {np.mean(similarities):.4f}")
        print(f"{t['single_median']}: {np.median(similarities):.4f}")

    return fig

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))

from src.etl.assign_point_to_geometry import read_point_data

gdf_2022 = read_point_data("../data/proc/embeddings/alpha_earth/cdmx/year=2022/", file_format="parquet").query("CVE_MUN == 3")
gdf_2023 = read_point_data("../data/proc/embeddings/alpha_earth/cdmx/year=2023/", file_format="parquet").query("CVE_MUN == 3")

In [ ]:
feature_cols = [f"A{str(i).zfill(2)}" for i in range(64)]

In [ ]:
len(gdf_2022)

In [ ]:
df = (
    gdf_2022
    .sjoin_nearest(
        gdf_2023, how = "left"
        )
)

gdf = gpd.GeoDataFrame(
    df
    , geometry=gpd.points_from_xy(df["lon_left"], df["lat_left"])
    , crs = gdf_2022.crs
)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

X = gdf[[x + "_left" for x in feature_cols]].to_numpy()
Y = gdf[[x + "_right" for x in feature_cols]].to_numpy()

gdf["cosine_similarity"] = [
    cosine_similarity(x.reshape(1, -1), y.reshape(1, -1))[0, 0]
    for x, y in zip(X, Y)
]

In [ ]:
for _lang in LANGS:
    fig = plot_sim_distribution_single(gdf["cosine_similarity"], lang=_lang)
    finish(fig, "similarity_between_years", _lang)

In [ ]:
import contextily as ctx

# Convert to Web Mercator for basemap + buffering
pd_gpd_3857 = pd_gpd.to_crs(epsg=3857)
gdf_3857 = gdf.to_crs(epsg=3857)

# Select low similarity points
low_sim = gdf_3857[gdf_3857.cosine_similarity <= 0.85].copy()

# Buffer points (meters)
low_sim["geometry"] = low_sim.geometry.buffer(100)  # 100 m buffer, adjust

for _lang in LANGS:
    t = TEXTS[_lang]

    # Plot AOI
    fig, ax = plt.subplots(figsize=(10, 7))

    pd_gpd_3857.plot(
        ax=ax,
        color="#04506e",
        alpha=0.2,
        edgecolor="#03304f",
        linewidth=2,
        zorder=1,
    )

    # Plot buffered low similarity clusters
    low_sim.plot(
        ax=ax,
        color="red",
        alpha=0.35,
        edgecolor="darkred",
        linewidth=0.5,
        zorder=3,
    )

    # Basemap
    ctx.add_basemap(
        ax,
        source=ctx.providers.CartoDB.Positron,
    )

    ax.set_axis_off()

    ax.set_title(
        t["lowsim_title"],
        fontsize=14,
        pad=0,
        loc="left",
    )

    finish(fig, "low_similarity_clusters", _lang)